In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import time
import os
from pathlib import Path
from Bio import SeqIO
from io import StringIO 
from Bio import Entrez
from Bio.Blast import NCBIWWW
from Bio.Blast import NCBIXML
from Bio import AlignIO
from Bio import Phylo
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor

Entrez.email = "muhammetcetin2025@gtu.edu.tr"
fasta_file = "/Users/azizcetin/Desktop/team_12/mystery_orfs.fasta"

In [2]:
#STEP1 - Load and inspect

def create_sanity_table(fasta_filepath):
    records_data = []
    for record in SeqIO.parse(fasta_filepath, "fasta"):
        protein_id = record.id
        sequence = str(record.seq)
        seq_length = len(sequence)
        first_30_aa = sequence[:30]
        records_data.append({
            "Protein_ID": protein_id,
            "Length": seq_length,
            "First_30_AA": first_30_aa
        })

    sanity_df = pd.DataFrame(records_data)
    sanity_df.index = sanity_df.index + 1 
    return sanity_df

fasta_file = "/Users/azizcetin/Desktop/team_12/mystery_orfs.fasta"

df_sanity = create_sanity_table(fasta_file)
    
print("--- PROTEIN SANITY TABLE ---")
print(df_sanity.to_string())


--- PROTEIN SANITY TABLE ---
   Protein_ID  Length                     First_30_AA
1     mORF_01     644  MGRIIGIDLGTTNSCVAVLDGDSAKVIENA
2     mORF_02     397  MAKEKFERSKPHVNVGTIGHVDHGKTTLTA
3     mORF_03     671  MNDMAKNLILWLVIAAVLLTVFNNFSTESA
4     mORF_04     806  MSEQAYDSSSIKVLKGLDAVRKRPGMYIGD
5     mORF_05     499  MPLFFSPERSFWVMYEKTLTQLAAALKSGE
6     mORF_06     346  MMSLKTMSQTAKTRVLTGITTSGTPHLGNY
7     mORF_07     308  MRFSANSFIAKPLIAACAVSLAAFSQVSAN
8     mORF_08     236  MSNLQVSVYRYNPETDSAPYMQEFQVDTKG
9     mORF_09     389  MTFGIPTKQLVRASLGALSLALLVGCASKG
10    mORF_10     341  MLSSDAPLPRLSTMHKLLTLCWLGLLTTVS
11    mORF_11     207  MKKLPSYCGMLLASALVLPVSTVAVADEVV
12    mORF_12     273  MPDATASSRIVPDVDQSLTAQHLRHRKGPT


In [5]:
# --- SAFE BASE DIRECTORY ---
BASE_DIR = "/Users/azizcetin/blast_results"
os.makedirs(BASE_DIR, exist_ok=True)

# --- SMART CACHE FUNCTION ---
def cached(out_path, fetch_fn):
    """
    Reads from cache if exists. If the cached file contains an NCBI error, 
    it ignores the cache, refetches the data, and overwrites it.
    """
    if Path(out_path).exists():
        text = Path(out_path).read_text()
        # Check if the cached file is actually an NCBI error page
        if "Message ID#" not in text and "Error:" not in text:
            return text
        else:
            print(f"  [Cache Override] Corrupted cache detected for {Path(out_path).name}. Refetching...")
    
    # Fetch fresh data from NCBI
    text = fetch_fn()
    
    # Save to disk only if it is a valid XML response (not an error)
    if "Message ID#" not in text and "Error:" not in text:
        Path(out_path).parent.mkdir(parents=True, exist_ok=True)
        Path(out_path).write_text(text)
        
    return text

# --- MAIN PIPELINE ---
def run_integrated_pipeline(fasta_filepath):
    records = list(SeqIO.parse(fasta_filepath, "fasta"))
    blast_results, hypothetical_records, psi_results = [], [], []
    
    print(f"Pipeline started for {len(records)} sequences...")
    
    # 1. Standard BLASTP
    for record in records:
        xml_file_path = f"{BASE_DIR}/blast_cache/{record.id}_blast.xml"
        
        try:
            def fetch_blast():
                handle = NCBIWWW.qblast("blastp", "nr", record.seq)
                res = handle.read()
                time.sleep(15) 
                return res

            xml_data = cached(xml_file_path, fetch_blast)
            blast_record = NCBIXML.read(StringIO(xml_data))
            
            if len(blast_record.alignments) == 0:
                hypothetical_records.append(record)
                continue
                
            best_alignment = blast_record.alignments[0]
            best_hsp = best_alignment.hsps[0]
            title = best_alignment.title
            
            blast_results.append({
                "Query_ID": record.id,
                "Top_Hit": title[:60] + "...",
                "E_Value": best_hsp.expect,
                "Identity_(%)": round((best_hsp.identities / best_hsp.align_length) * 100, 2)
            })
            
            # Filter logic: catches 'hypothetical' or 'unknown'
            if "hypothetical" in title.lower() or "unknown" in title.lower():
                hypothetical_records.append(record)
                
        except Exception as e:
            print(f"ERROR ({record.id} Standard BLAST): {e}")

    df_blast = pd.DataFrame(blast_results)

    # 2. PSI-BLAST for remaining hypothetical proteins
    if hypothetical_records:
        print(f"\nInitiating PSI-BLAST for {len(hypothetical_records)} hypothetical proteins...")
        
        for record in hypothetical_records:
            xml_file_path = f"{BASE_DIR}/psiblast_cache/{record.id}_psiblast.xml"
            
            try:
                def fetch_psiblast():
                    # FIXED: Added word_size=3 explicitly for PSI-BLAST validation
                    handle = NCBIWWW.qblast("blastp", "nr", record.seq, service="psi", word_size=3)
                    res = handle.read()
                    time.sleep(15) 
                    return res

                xml_data = cached(xml_file_path, fetch_psiblast)
                
                # Double check if the API still returned an error string
                if "Message ID#" in xml_data:
                    print(f"ERROR ({record.id} PSI-BLAST): NCBI API refused the request. Passing...")
                    continue

                blast_record = NCBIXML.read(StringIO(xml_data))
                
                if len(blast_record.alignments) == 0:
                    continue
                    
                best_alignment = blast_record.alignments[0]
                best_hsp = best_alignment.hsps[0]
                
                psi_results.append({
                    "Query_ID": record.id,
                    "PSI_Top_Hit": best_alignment.title[:60] + "...",
                    "PSI_E_Value": best_hsp.expect,
                    "PSI_Identity_(%)": round((best_hsp.identities / best_hsp.align_length) * 100, 2)
                })
                print(f"  -> SUCCESS ({record.id}): Found remote homolog via PSI-BLAST.")
                
            except Exception as e:
                print(f"ERROR ({record.id} PSI-BLAST): {e}")

    df_psi = pd.DataFrame(psi_results)
    return df_blast, df_psi

# --- EXECUTION ---
fasta_file = "/Users/azizcetin/Desktop/team_12/mystery_orfs.fasta" 
df_standard, df_psi = run_integrated_pipeline(fasta_file)

# Save the final dataframes to the safe directory
df_standard.to_csv(f"{BASE_DIR}/1_standard_blast_results.tsv", sep="\t", index=False)
if not df_psi.empty:
    df_psi.to_csv(f"{BASE_DIR}/2_psiblast_hypothetical_results.tsv", sep="\t", index=False)

print(f"\nPipeline finished! Output saved in '{BASE_DIR}'.")

if not df_psi.empty:
    print("\n--- PSI-BLAST Highlights ---")
    print(df_psi.head(12))

Pipeline started for 12 sequences...

Initiating PSI-BLAST for 4 hypothetical proteins...
  -> SUCCESS (mORF_06): Found remote homolog via PSI-BLAST.
  -> SUCCESS (mORF_10): Found remote homolog via PSI-BLAST.
  -> SUCCESS (mORF_11): Found remote homolog via PSI-BLAST.
  -> SUCCESS (mORF_12): Found remote homolog via PSI-BLAST.

Pipeline finished! Output saved in '/Users/azizcetin/blast_results'.

--- PSI-BLAST Highlights ---
  Query_ID                                        PSI_Top_Hit    PSI_E_Value  \
0  mORF_06  gb|ELY21290.1| Tryptophanyl-tRNA synthetase, c...   0.000000e+00   
1  mORF_10  gb|ELY22254.1| hypothetical protein HALTITAN_0...   0.000000e+00   
2  mORF_11  ref|WP_009286036.1| hypothetical protein [Vree...  2.974520e-128   
3  mORF_12  ref|WP_009286699.1| MULTISPECIES: hypothetical...  1.499080e-144   

   PSI_Identity_(%)  
0             100.0  
1             100.0  
2             100.0  
3             100.0  


In [7]:
def generate_final_annotation_table():
    BASE_DIR = "/Users/azizcetin/blast_results"
    FASTA_FILE = "/Users/azizcetin/Desktop/team_12/mystery_orfs.fasta"
    
    print("Generating the final assignment annotation table...")

    # 1. Parse sequence lengths
    seq_lengths = {}
    try:
        for record in SeqIO.parse(FASTA_FILE, "fasta"):
            seq_lengths[record.id] = len(record.seq)
    except FileNotFoundError:
        print(f"ERROR: Cannot find {FASTA_FILE}")
        return

    # 2. Load BLAST and PSI-BLAST data
    try:
        df_blast = pd.read_csv(f"{BASE_DIR}/1_standard_blast_results.tsv", sep="\t")
    except FileNotFoundError:
        print("ERROR: Standard BLAST results not found.")
        return

    try:
        df_psi = pd.read_csv(f"{BASE_DIR}/2_psiblast_hypothetical_results.tsv", sep="\t")
    except FileNotFoundError:
        df_psi = pd.DataFrame()

    # 3. Build the rows according to Step 4 requirements
    rows = []

    for _, b_row in df_blast.iterrows():
        orf_id = b_row['Query_ID']
        length = seq_lengths.get(orf_id, 0)
        blast_top_hit = b_row['Top_Hit']
        blast_evalue = float(b_row['E_Value'])
        blast_pct_id = b_row['Identity_(%)']
        
        # Placeholders for Pfam (to be filled manually or in subsequent steps)
        pfam_hits = ""
        pfam_evalue = ""
        
        twilight_used = "None"
        twilight_outcome = "N/A"
        
        is_hypothetical = "hypothetical" in blast_top_hit.lower() or "unknown" in blast_top_hit.lower()
        
        # Check if it went through the Twilight Zone (PSI-BLAST)
        psi_hit_info = None
        if is_hypothetical and not df_psi.empty and orf_id in df_psi['Query_ID'].values:
            psi_match = df_psi[df_psi['Query_ID'] == orf_id].iloc[0]
            twilight_used = "PSI-BLAST"
            twilight_outcome = f"Hit: {psi_match['PSI_Top_Hit']} (E: {psi_match['PSI_E_Value']})"
            psi_hit_info = psi_match
            
        # Determine Proposed Annotation, Evidence, and Confidence Score
        if not is_hypothetical:
            proposed_annotation = blast_top_hit.split("[")[0].strip() # Clean up the name a bit
            evidence_summary = "Strong standard BLAST match to well-characterized protein"
            
            if blast_evalue < 1e-50:
                confidence = "High"
            elif blast_evalue < 1e-10:
                confidence = "Medium"
            else:
                confidence = "Low"
                
        else:
            if twilight_used == "PSI-BLAST" and psi_hit_info is not None:
                psi_is_hypothetical = "hypothetical" in psi_hit_info['PSI_Top_Hit'].lower()
                
                if not psi_is_hypothetical and float(psi_hit_info['PSI_E_Value']) < 1e-10:
                    proposed_annotation = psi_hit_info['PSI_Top_Hit'].split("[")[0].strip()
                    evidence_summary = "Converged on characterized family via PSI-BLAST"
                    confidence = "Medium"
                else:
                    proposed_annotation = "Unknown / Hypothetical Protein"
                    evidence_summary = "Remains hypothetical even after PSI-BLAST profiling"
                    confidence = "Low"
            else:
                proposed_annotation = "Unknown / Hypothetical Protein"
                evidence_summary = "Hypothetical in standard BLAST, no further twilight data"
                confidence = "Low"

        # 4. Append exact required columns
        rows.append({
            "orf_id": orf_id,
            "length": length,
            "blast_top_hit": blast_top_hit,
            "blast_evalue": blast_evalue,
            "blast_pct_id": blast_pct_id,
            "pfam_hits": pfam_hits,
            "pfam_evalue": pfam_evalue,
            "twilight_followup_used": twilight_used,
            "twilight_followup_outcome": twilight_outcome,
            "proposed_annotation": proposed_annotation,
            "evidence_summary": evidence_summary,
            "confidence": confidence
        })

    # Export to CSV
    df_out = pd.DataFrame(rows)
    out_path = f"{BASE_DIR}/Team_12_annotation_table.csv"
    df_out.to_csv(out_path, index=False)
    
    print(f"Success! Saved to {out_path}")
    return df_out

# --- EXECUTION ---
df_final = generate_final_annotation_table()

if df_final is not None:
    print("\n--- Final Output Preview ---")
    print(df_final[['orf_id', 'proposed_annotation', 'confidence', 'twilight_followup_used']].head(12))

Generating the final assignment annotation table...
Success! Saved to /Users/azizcetin/blast_results/Team_12_annotation_table.csv

--- Final Output Preview ---
     orf_id                                proposed_annotation confidence  \
0   mORF_01  ref|WP_009287914.1| MULTISPECIES: molecular ch...       High   
1   mORF_02  ref|WP_009288143.1| MULTISPECIES: elongation f...       High   
2   mORF_03  ref|WP_009287922.1| MULTISPECIES: ATP-dependen...       High   
3   mORF_04  ref|WP_009286362.1| MULTISPECIES: DNA topoisom...       High   
4   mORF_05                             gb|ELY21083.1| Amidase       High   
5   mORF_06  gb|ELY21290.1| Tryptophanyl-tRNA synthetase, c...       High   
6   mORF_07  ref|WP_009287804.1| MULTISPECIES: glycine beta...       High   
7   mORF_08  ref|WP_009287296.1| MULTISPECIES: succinate de...       High   
8   mORF_09  ref|WP_009288403.1| MULTISPECIES: outer membra...       High   
9   mORF_10                     Unknown / Hypothetical Protein        